In [1]:
import pandas as pd

df_retail = pd.read_excel(
    "../data/raw/Online Retail.xlsx",
    engine="openpyxl"
)

print(f"Registros originales: {df_retail.shape[0]}")

Registros originales: 541909


In [20]:
df_trabajo = df_retail.copy()

In [3]:
df_trabajo["InvoiceNo"].head(10)

0    536365
1    536365
2    536365
3    536365
4    536365
5    536365
6    536365
7    536366
8    536366
9    536367
Name: InvoiceNo, dtype: object

In [4]:
df_trabajo["EsCancelacion"] = (
    df_trabajo["InvoiceNo"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

display(df_trabajo[["InvoiceNo", "EsCancelacion"]].head())

,InvoiceNo,EsCancelacion
0,536365,False
1,536365,False
2,536365,False
3,536365,False
4,536365,False


In [5]:
total_cancelaciones = df_trabajo["EsCancelacion"].sum()

print(f"Líneas de cancelación: {total_cancelaciones}")

Líneas de cancelación: 9288


In [6]:
df_trabajo["CantidadNegativa"] = df_trabajo["Quantity"] < 0

df_trabajo["PrecioNoPositivo"] = df_trabajo["UnitPrice"] <= 0

In [7]:
print("Cantidades negativas:")
print(df_trabajo["CantidadNegativa"].sum())

print("Precios no positivos:")
print(df_trabajo["PrecioNoPositivo"].sum())

Cantidades negativas:
10624
Precios no positivos:
2517


In [8]:
df_trabajo["InvoiceDate"] = pd.to_datetime(
    df_trabajo["InvoiceDate"],
    errors="coerce"
)

In [9]:
print(df_trabajo["InvoiceDate"].dtype)

print(df_trabajo["InvoiceDate"].isna().sum())

datetime64[ns]
0


In [10]:
df_trabajo["Anio"] = df_trabajo["InvoiceDate"].dt.year

df_trabajo["Mes"] = df_trabajo["InvoiceDate"].dt.month

In [11]:
df_ventas = df_trabajo[
    (~df_trabajo["EsCancelacion"]) &
    (df_trabajo["Quantity"] > 0) &
    (df_trabajo["UnitPrice"] > 0) &
    (df_trabajo["InvoiceDate"].notna())
].copy()

display(df_ventas.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,EsCancelacion,CantidadNegativa,PrecioNoPositivo,Anio,Mes
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,2010,12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,2010,12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12


In [12]:
df_ventas["ImporteLinea"] = (
    df_ventas["Quantity"] * df_ventas["UnitPrice"]
)

display(df_ventas.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,EsCancelacion,CantidadNegativa,PrecioNoPositivo,Anio,Mes,ImporteLinea
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,2010,12,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,2010,12,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,2010,12,20.34


In [13]:
importe_total = df_ventas["ImporteLinea"].sum()

print(f"Importe total seleccionado: {importe_total:,.2f} £")

Importe total seleccionado: 10,666,684.54 £


In [14]:
diferencia_de_dimensiones = df_retail.shape[0] - df_ventas.shape[0]

display(diferencia_de_dimensiones)

11805

In [16]:
customers_ausentes = df_ventas["CustomerID"].isna().sum()

display(customers_ausentes)

np.int64(132220)

In [17]:
importe_acumulado_pais = df_ventas.groupby("Country")["ImporteLinea"].sum().sort_values(ascending=False)

display(importe_acumulado_pais)

Country
United Kingdom          9025222.084
Netherlands              285446.340
EIRE                     283453.960
Germany                  228867.140
France                   209715.110
Australia                138521.310
Spain                     61577.110
Switzerland               57089.900
Belgium                   41196.340
Sweden                    38378.330
Japan                     37416.370
Norway                    36165.440
Portugal                  33747.100
Finland                   22546.080
Singapore                 21279.290
Channel Islands           20450.440
Denmark                   18955.340
Italy                     17483.240
Hong Kong                 15691.800
Cyprus                    13590.380
Austria                   10198.680
Israel                     8135.260
Poland                     7334.650
Greece                     4760.520
Unspecified                4749.790
Iceland                    4310.000
Canada                     3666.380
USA                 

In [19]:
lineas_mes = df_ventas.groupby("Anio")["Mes"].value_counts()

display(lineas_mes)

Anio  Mes
2010  12     41480
2011  11     83369
      10     59304
      9      49261
      7      38645
      5      36164
      6      35977
      3      35803
      8      34483
      1      34306
      4      29096
      2      27105
      12     25111
Name: count, dtype: int64

In [21]:
df_ventas.to_csv(
    "../data/processed/online_retail_ventas.csv",
    index=False
)

print("Archivo CSV exportado correctamente")

Archivo CSV exportado correctamente
